Created by Angelo Orletti Del Rey for its masters in psychiatry at Unifesp - SP - Brasil

It was based in our groups previeous works in cotical striatal connectivity in INPD data base of fmri

In [1]:
import numpy as np
import os, sys
import bids
import nibabel as nib
from scipy.stats import pearsonr
from scipy.stats import chi2
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families import NegativeBinomial
from nilearn import plotting as nplot
from nilearn import image as nimg
from nilearn.image import resample_to_img
import matplotlib.pyplot as plt
import argparse
import pandas as pd

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from scipy.linalg import LinAlgError

# Suppress convergence warnings
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")


c:\Users\angel\Documents\masters-uploaded-github\INPD-neuroimage-cape\.venv-CAPE-neuroimag\lib\site-packages\nilearn\__init__.py:67: FutureWarning: Python 3.7 support is deprecated and will be removed in release 0.12 of Nilearn. Consider switching to Python 3.9 or 3.10.
  _python_deprecation_warnings()


In [10]:
def parse():

        options = argparse.ArgumentParser(description="Run 1st level analysis. Created by ...")
        options.add_argument('-p', '--participants', nargs='+',dest="participants", action='store', type=str, required=False,
                            help='id of subject or list of subjects')
        options.set_defaults(participants=None)
        options.add_argument('-w', '--workdir',dest="workdir", action='store', type=str, required=False,
                            help='the work directory for the project')
        options.set_defaults(workdir=os.environ["ROOTDIR"])
        options.add_argument('-b', '--bidsdir',dest="rawdata", action='store', type=str, required=False,
                            help='the work directory for the project')
        options.set_defaults(rawdata=os.path.join(os.environ["ROOTDIR"],"BIDS"))
        options.add_argument('-d', '--derivatives',dest="derivatives", action='store', type=str, required=True,
                            help='path to fMRIprep directory')
        options.add_argument('-o', '--outputs',dest="output", action='store', type=str, required=True,
                            help='path to fMRIprep directory')
        #print(options.parse_args())

        return options.parse_args()

os.environ["ROOTDIR"] = r'D://'   # seth path
rootdir = os.environ["ROOTDIR"]
if hasattr(sys, "ps1"):
    options = {}
    workdir = os.environ["ROOTDIR"]
    firstleveldir  = os.path.join(workdir,"BIDS","derivatives","first_level_results")
    demographic = os.path.join(workdir,"metadata")
    confounds = os.path.join(workdir,"metadata")
    output = os.path.join(workdir,'second_level_results')
    participants = []

else :
    options = parse()
    participants = options.participants
    workdir = options.workdir
    rawdata = options.rawdata
    derivat = options.derivatives
    output  = options.outputs

print('firstlevel: ', firstleveldir)
    
bidslayout = bids.BIDSLayout(firstleveldir, validate = False) #With validate = True it doesn't find any subjects

if not participants:
    participants = bidslayout.get_subjects() #take all subjects
    #participants_sub = ["sub-" + item for item in participants]

seednames = ['DCPutamen',
            'DorsalCaudate',
            'DRPutamen',
            'InfVentralCaudate',
            'SupVentralCaudate',
            'VRPutamen'
            ]

tmap_allsubj = [] #initialize array for tmaps

print("Loading tmaps for all participants...")
for p in participants:
    #print(f"Subject: {p}")
    p = p.replace("sub-", "")

    # for ses in bidslayout.get_sessions(subject=p):
    #     #print(f"Session: {ses}")                
    
    ses = '1'

    for r in bidslayout.get_runs(subject=p, session=ses):
        #print(f"Run: {r}")

        tmap_metadata = {
            'task': 'rest',
            'suffix': 'tstat',
            'extension': '.txt',
            'run': str(r),
            'session': str(ses),
            'subject': 'sub-'+p,
            'space': 'MNI152NLin2009cAsym',
            'seed': 'SeedtoROI'
        }

        # Save each tmap as a nifti file
        filepath = os.path.join(firstleveldir,tmap_metadata["subject"],"ses-"+tmap_metadata['session'],"func")
        filename = tmap_metadata["subject"] + "_" + \
                    "ses-"+tmap_metadata['session'] + "_" + \
                    "task-"+tmap_metadata['task'] + "_" + \
                    "run-"+tmap_metadata['run'] + "_" + \
                    "space-"+tmap_metadata['space'] + "_" + \
                    "seed-"+tmap_metadata['seed'] + "_" + \
                    tmap_metadata["suffix"] + \
                    tmap_metadata["extension"]
        tmap_path = os.path.join(filepath, filename)

        #load the tmaps for each subject and then append to the tmap_allsubj array
        tmap = pd.read_csv(tmap_path, delimiter=' ', header=None, skiprows=1) #also skips the first row
        tmap.columns = seednames
        tmap_allsubj.append(tmap)

#load socioeconomic and psichometric data for all subjects
print("Loading socioeconomic and psichometric data for all participants...")
socio_psi=pd.read_csv(os.path.join(demographic,'variaveis_analise_conf_nova-exclusão.tsv'), sep='\t')
#substitute in age colunm , to . for decimals if necessary and convert to float
# socio_psi['age'] = socio_psi['age'].str.replace(',', '.').astype(float)

# organinzing the tmaps. Joining the columns of the tmaps for all subjects. So for each subject
# we have to join the first collunm of the first tmap with the first column of the second tmap and so on
# and then join the second column of the first tmap with the second column of the second tmap and so on
# and so on
print("Organizing tmaps...")
tmap_allsubj = pd.concat(tmap_allsubj, axis=1)
tmap_allsubj_organized = {}

for seed in seednames:
    #drop columns with names different from seednames[i]
    tmap_allsubj_organized[seed] = tmap_allsubj.drop(columns=[col for col in tmap_allsubj.columns if col != seed])

# What do we have now: tmap_allsubj_organized is a dictionary with keys being the seednames and values being
# dataframes with the tmaps for all subjects for that seed. This have dimensions of n_ROIs x n_subjects
# Now we have to transpose it to get the data in the format n_subjects x n_ROIs and then join with the
# socioeconomic and psichometric data
# Also, we have to add the sub_id column to the conectivity data to be able to join with the socioeconomic
# and psichometric data
subs = {'sub_id': []}
for p in participants:
    ses = '1'
    if bidslayout.get_runs(subject=p, session=ses) != []:
        subs['sub_id'].append('sub-'+p)

subs = pd.DataFrame(subs)

# create the collunms names for the data frame of rois
roi_names = [f'roi_{i}' for i in range(101)]

for seed in seednames:
    tmap_allsubj_organized[seed] = tmap_allsubj_organized[seed].transpose()
    tmap_allsubj_organized[seed].columns = roi_names
    tmap_allsubj_organized[seed] = tmap_allsubj_organized[seed].reset_index().join(subs, how='inner')
    tmap_allsubj_organized[seed] = tmap_allsubj_organized[seed].drop(columns=['index'])

# Now we have to join the tmaps with the socioeconomic and psichometric data
print("Joining tmaps with socioeconomic and psichometric data...")

for seed in seednames:
    tmap_allsubj_organized[seed] = pd.merge(tmap_allsubj_organized[seed], socio_psi, on='sub_id', how='inner')
    # tmap_allsubj_organized[seed] = tmap_allsubj_organized[seed].drop(columns=['Unnamed: 0'])


firstlevel:  D://BIDS\derivatives\first_level_results
Loading tmaps for all participants...
Loading socioeconomic and psichometric data for all participants...
Organizing tmaps...
Joining tmaps with socioeconomic and psichometric data...


In [ ]:
# now we run the regression analysis for each seed. The dependent variable is the connectivity and the independent
# variables are the CAPE scores. The socioeconomic variables are used as confund variables
print("Running regression analysis (salience and defualt mode network)...")

shaeffer_network = pd.read_csv(os.path.join(demographic, 'tpl-MNI152NLin2009cAsym_atlas-Schaefer2018_desc-100Parcels7Networks_dseg.tsv'), delimiter='\t')

#extract the index with the name with SalVentAttn - salience network
salience_index = shaeffer_network[shaeffer_network['name'].str.contains('SalVentAttn')]
#extract the index with the name with Default - default mode network
default_index = shaeffer_network[shaeffer_network['name'].str.contains('Default')]

# running the regression analysis for each roi in the salience network
results_salience, results_default = {}, {}

for seed in seednames:
    results_salience[seed], results_default[seed] = {}, {}

    #salience analisys
    for roi in salience_index['index']:

        #building my strign for the fomrula
        formula_salience = f'cape_tot ~ roi_{roi} + age + C(gender) + abepscore + C(colection_site) + fd_mean_value'
        
        #running the regression analysis
        model_salience = smf.glm(formula_salience, data=tmap_allsubj_organized[seed],
                                  family=NegativeBinomial())
        fit_salience = model_salience.fit()

        # Extract desired values
        results_salience[seed][roi] = {
            'coef': fit_salience.params[f'roi_{roi}'],
            'intercept': fit_salience.params['Intercept'],
            't_value': fit_salience.tvalues[f'roi_{roi}'],
            'p_value': fit_salience.pvalues[f'roi_{roi}'],
            # 'R_squared': fit_salience.rsquared,
            # 'Adj_R_squared': fit_salience.rsquared_adj,
            'AIC': fit_salience.aic,
            'BIC': fit_salience.bic
        }
        
    #DMN analisys
    for roi in default_index['index']:

        #building my strign for the fomrula
        formula_default = f'cape_tot ~ roi_{roi} + age + C(gender) + abepscore + C(colection_site) + fd_mean_value'
        
        # Run the regression analysis
        model_default = smf.glm(formula_default, data=tmap_allsubj_organized[seed],
                                  family=NegativeBinomial())
        fit_default = model_default.fit()

        # Extract desired values
        results_default[seed][roi] = {
            'coef': fit_default.params[f'roi_{roi}'],
            'intercept': fit_default.params['Intercept'],
            't_value': fit_default.tvalues[f'roi_{roi}'],
            'p_value': fit_default.pvalues[f'roi_{roi}'],
            # 'R_squared': fit_default.rsquared,
            # 'Adj_R_squared': fit_default.rsquared_adj,
            'AIC': fit_default.aic,
            'BIC': fit_default.bic
        }

Running regression analysis (salience and defualt mode network)...


c:\Users\angel\Documents\masters-uploaded-github\INPD-neuroimage-cape\.venv-CAPE-neuroimag\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1809: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
  FutureWarning
c:\Users\angel\Documents\masters-uploaded-github\INPD-neuroimage-cape\.venv-CAPE-neuroimag\lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1809: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of

In [47]:
#select a line to see the pvalues of the last fit

# fit_salience.summary()
results_default['DorsalCaudate']

{38: {'coef': -0.032797194700302214,
  'intercept': 1.501281403723897,
  't_value': -1.0996026856135626,
  'p_value': 0.27150527133067714,
  'AIC': 2166.398240909675,
  'BIC': -1926.86248929265},
 39: {'coef': -0.006592576532798097,
  'intercept': 1.4764799487702303,
  't_value': -0.2623614281422042,
  'p_value': 0.7930428072329092,
  'AIC': 2167.549917808258,
  'BIC': -1925.7108123940668},
 40: {'coef': -0.02698148581914448,
  'intercept': 1.494506288114588,
  't_value': -0.8788865354615296,
  'p_value': 0.37946279989980625,
  'AIC': 2166.8128997404583,
  'BIC': -1926.4478304618667},
 41: {'coef': 0.0022456696229460477,
  'intercept': 1.460737268192114,
  't_value': 0.08506732766115646,
  'p_value': 0.9322078648749083,
  'AIC': 2167.612072148631,
  'BIC': -1925.6486580536944},
 42: {'coef': 0.04031252039286244,
  'intercept': 1.362844902411607,
  't_value': 1.283006347961416,
  'p_value': 0.1994898521072751,
  'AIC': 2165.939625858856,
  'BIC': -1927.321104343469},
 43: {'coef': 0.002

In [48]:
#correcting the p-values
from statsmodels.stats.multitest import multipletests

for seed in seednames:
    print(f'Salience - {seed}')
    p_values = []

    for roi in results_salience[seed].keys():
        # print(roi)
        # Extract the p-values for the current seed and roi
        p_values.append(results_salience[seed][roi]['p_value'])

    #correct the p-values using the Benjamini-Hochberg method
    corrected_p_values = multipletests(p_values, method='fdr_bh')[1]

    # Update the results with the corrected p-values
    for index, roi in enumerate(results_salience[seed].keys()):
        results_salience[seed][roi]['corrected_p_value'] = corrected_p_values[index]

    print(f'DMN - {seed}')
    p_values = []

    for roi in results_default[seed].keys():
        # print(roi)
        # Extract the p-values for the current seed and roi
        p_values.append(results_default[seed][roi]['p_value'])

    #correct the p-values using the Benjamini-Hochberg method
    corrected_p_values = multipletests(p_values, method='fdr_bh')[1]

    # Update the results with the corrected p-values
    for index, roi in enumerate(results_default[seed].keys()):
        # print(f'index {index} roi {roi}')
        results_default[seed][roi]['corrected_p_value'] = corrected_p_values[index]

Salience - DCPutamen
DMN - DCPutamen
Salience - DorsalCaudate
DMN - DorsalCaudate
Salience - DRPutamen
DMN - DRPutamen
Salience - InfVentralCaudate
DMN - InfVentralCaudate
Salience - SupVentralCaudate
DMN - SupVentralCaudate
Salience - VRPutamen
DMN - VRPutamen


In [ ]:
for seed in seednames:
    for roi in results_salience[seed].keys():
        #transfor resultes_salience[seed][roi] to a dataframe
        results_salience[seed][roi] = pd.DataFrame(results_salience[seed][roi],index=[0])
        results_salience[seed][roi].to_csv(os.path.join(output, f'{seed}_roi-{roi}_2ndlvl.csv'))
    for roi in results_default[seed].keys():
        results_default[seed][roi].to_csv(os.path.join(output, f'{seed}_roi-{roi}_2ndlvl.csv'))
    tmap_allsubj_organized[seed].to_csv(os.path.join(output,f'{seed}_data-2ndlvl.csv'))

Exporting results to csv files...


KeyError: 'cape_tot'

In [49]:
#evaluating p corrected <= 0.05

for cape in ['cape_tot', 'cape_PA_score', 'cape_PI_score', 'cape_BE_score']:
    for seed in seednames:
        print(f'------ DMN seed-{seed} {cape} ------')
        for roi in results_default[cape][seed].keys():
            if results_default[cape][seed][roi]['corrected_p_value'].iloc[0] <= 0.05:
                print(f'{cape} roi-{roi} -> corr_p_val {results_default[cape][seed][roi]['corrected_p_value'].iloc[0]}')
        
    for seed in seednames:
        print(f'------- Salience seed-{seed} {cape} -------')
        for roi in results_salience[cape][seed].keys():
            if results_salience[cape][seed][roi]['corrected_p_value'].iloc[0] <=0.05:
                print(f'{cape} roi-{roi} -> corr_p_val {results_salience[cape][seed][roi]['corrected_p_value'].iloc[0]}')

SyntaxError: invalid syntax (3413645004.py, line 8)

In [6]:
tmap_allsubj_organized['DorsalCaudate']

,roi_0,roi_1,roi_2,roi_3,roi_4,roi_5,roi_6,roi_7,roi_8,roi_9,...,sub_id,gender,colection_site,abepscore,age,fd_count_high_value,cape_PA_score,cape_PI_score,cape_BE_score,cape_tot
0,5.166577,6.130344,-1.510647,0.268063,-2.826788,0.006392,1.070554,2.120269,-2.167125,-4.632179,...,sub-00003,2,1,10,8.361396,3,0,1,0,1
1,1.630589,0.051181,-0.363106,-1.544317,-0.784023,0.005769,-1.340514,-2.755960,-2.160502,-0.989861,...,sub-00015,2,1,19,12.221766,3,0,4,0,5
2,-1.864590,1.379442,-0.910317,-0.307140,-0.521040,-0.005039,-1.853937,0.519793,-2.330436,-1.851758,...,sub-00019,2,1,13,11.049966,17,0,2,0,4
3,-0.251798,-0.508129,1.549115,1.495391,-0.923917,0.020160,-0.834836,-1.102018,-0.796371,-0.602144,...,sub-00022,2,1,12,13.475702,1,0,3,0,6
4,3.423935,4.543147,2.775583,-0.921911,0.945897,-0.021527,1.980327,5.242491,2.538129,-0.200997,...,sub-00028,1,1,16,7.556468,6,0,1,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
407,1.795409,1.649689,-1.982212,-0.467509,0.888754,-0.032821,-0.022741,-0.306769,-0.180652,-2.430632,...,sub-02494,2,2,18,7.225188,3,0,2,0,3
408,3.188942,5.285810,0.397248,0.589257,-0.114454,-0.002135,3.686344,0.668939,-1.507971,-0.612246,...,sub-02496,1,2,17,8.038330,21,2,5,3,12
409,2.454290,0.818743,-1.054593,-2.489692,-1.381791,-0.865129,-1.279167,-0.308884,-0.672654,-0.672592,...,sub-02502,1,2,23,9.911020,16,0,2,0,6
410,2.229099,1.547304,0.462994,2.294430,4.360636,-0.005273,2.841847,3.796825,2.854375,0.717036,...,sub-02507,1,2,19,8.440794,8,0,1,1,4
